In [2]:
# 🍬 WishWhisk V1: 我的心愿雷达配置

# wishlist 是一个列表 [ ]，里面装着我们所有的任务卡片 { }
wishlist = [
    {
        "name": "Lemaire scarf bag", 
        "keywords": ["lemaire", "scarf", "black"], 
        "target_price": 1350  
    },
    {
        "name": "Lemaire large croissant bag dark chocolate",
        "keywords": ["lemaire", "croissant", "large", "brown"], 
        "target_price": 1400 
    },
    
]

print(f"✅ Successfully loaded {len(wishlist)} tracking tasks! Ready to hunt.")

✅ Successfully loaded 2 tracking tasks! Ready to hunt.


In [3]:
import os

def notify_mac(title, text):
    # Use os.system to trigger macOS's built-in AppleScript for notifications
    os.system(f"""osascript -e 'display notification "{text}" with title "{title}"'""")

# Test it out right away!
notify_mac("TreatTracker 🍬", "Deal alert! Your Lemaire Croissant dropped below $1000. Go get it!")

In [4]:
def build_search_url(keywords):
    # Join the keywords using a plus sign "+"
    search_query = "+".join(keywords)
    
    # Combine the base URL with our query
    full_url = f"https://www.ssense.com/en-ca/women?q={search_query}"
    
    return full_url

# Let's test if our machine works!
test_keywords = ["lemaire", "croissant", "small", "black"]
generated_url = build_search_url(test_keywords)

print("Generated URL:")
print(generated_url)

Generated URL:
https://www.ssense.com/en-ca/women?q=lemaire+croissant+small+black


In [7]:
# We import the 'requests' from our newly installed elite toolkit
from curl_cffi import requests

def fetch_webpage(url):
    print(f"Sending the elite agent to: {url} ...")
    
    # The magic happens here: impersonate="chrome110" 
    # This perfectly mimics the exact walking posture of a real Chrome browser!
    response = requests.get(url, impersonate="chrome110")
    
    if response.status_code == 200:
        print("✅ Success! The security guard was fooled. We got the data.")
        print("-" * 30)
        print("Here is a peek at the HTML blueprint:")
        print(response.text[:300]) 
        print("-" * 30)
        return response.text
    elif response.status_code == 403:
        print("🛑 Oh no! The guard STILL kicked us out (403 Forbidden).")
        return None
    else:
        print(f"⚠️ Something weird happened. Status Code: {response.status_code}")
        return None

# Let's test it again with our URL!
test_keywords = ["lemaire", "croissant", "small", "black"]
test_url = build_search_url(test_keywords)
html_data = fetch_webpage(test_url)

Sending the elite agent to: https://www.ssense.com/en-ca/women?q=lemaire+croissant+small+black ...
✅ Success! The security guard was fooled. We got the data.
------------------------------
Here is a peek at the HTML blueprint:
<!DOCTYPE html><html lang="en-ca"><head><meta name="language" content="en"><meta http-equiv="Content-Type" content="text/html; charset=utf-8"><title>Designer Clothes, Shoes & Bags for Women | SSENSE Canada</title><meta name="viewport" content="width=device-width,initial-scale=1,maximum-scale=1,user-
------------------------------


In [8]:
# We need BeautifulSoup to read the HTML blueprint
from bs4 import BeautifulSoup

def extract_price(html_data):
    print("🥣 Giving the blueprint to BeautifulSoup...")
    # 1. Turn the messy text into a searchable tree
    soup = BeautifulSoup(html_data, 'html.parser')
    
    # 2. Tell it to find the exact nametag you discovered!
    price_box = soup.find('span', attrs={'data-test': 'pdpRegularPriceText'})
    
    if price_box:
        # 3. We found the box! Now let's extract the text inside
        raw_text = price_box.text.strip()
        print(f"🎯 Jackpot! Found the price text: {raw_text}")
        

        # 1. 挨个把碍眼的符号替换成“空”（也就是删掉）
        clean_text = raw_text.replace('$', '').replace(',', '').replace('CAD', '').strip()

        # 2. 把清理干净的纯净文本，转换成带小数点的数学数字 (float)
        clean_price = float(clean_text)

        print(f"🧹 Cleaned up price with decimals: {clean_price}")
        return clean_price
    else:
        print("⚠️ Couldn't find the exact price box.")
        print("   (Note: We might be on a search list page instead of a single product page!)")
        return None

# Let's test our extractor with the html_data from the previous step!
if html_data:
    current_price = extract_price(html_data)

🥣 Giving the blueprint to BeautifulSoup...
⚠️ Couldn't find the exact price box.
   (Note: We might be on a search list page instead of a single product page!)


In [9]:
from bs4 import BeautifulSoup

def extract_prices_from_list(html_data):
    print("🥣 Giving the blueprint to BeautifulSoup...")
    soup = BeautifulSoup(html_data, 'html.parser')
    
    # 1. Use 'find_all' to get EVERY price box on the shelf!
    price_boxes = soup.find_all('span', attrs={'data-test': 'productCurrentPrice1'})
    
    if not price_boxes:
        print("⚠️ Couldn't find any prices. The shelf might be empty.")
        return []
        
    print(f"🎯 Jackpot! Found {len(price_boxes)} prices on this page.")
    
    cleaned_prices = []
    
    # 2. Loop through each box we found
    for box in price_boxes:
        raw_text = box.text.strip()
        
        # Your brilliant cleaning logic!
        clean_text = raw_text.replace('$', '').replace(',', '').replace('CAD', '').strip()
        
        if clean_text:
            clean_price = float(clean_text)
            cleaned_prices.append(clean_price)
            print(f"🛍️ Found a price: ${clean_price}")
            
    return cleaned_prices

# Let's test it with the html_data!
if html_data:
    all_prices = extract_prices_from_list(html_data)
    if all_prices:
        print("-" * 30)
        print(f"✨ The lowest price on this page is: ${min(all_prices)}")

🥣 Giving the blueprint to BeautifulSoup...
🎯 Jackpot! Found 1 prices on this page.
🛍️ Found a price: $775.0
------------------------------
✨ The lowest price on this page is: $775.0


In [11]:
from bs4 import BeautifulSoup

def extract_items_from_list(html_data):
    print("🥣 Giving the blueprint to BeautifulSoup...")
    soup = BeautifulSoup(html_data, 'html.parser')
    
    # 1. Get ALL name boxes and ALL price boxes
    name_boxes = soup.find_all('span', attrs={'data-test': 'productName1'})
    price_boxes = soup.find_all('span', attrs={'data-test': 'productCurrentPrice1'})
    
    # Quick check to make sure the page isn't empty
    if not name_boxes or not price_boxes:
        print("⚠️ Couldn't find items on the shelf.")
        return []
        
    print(f"🎯 Jackpot! Found {len(name_boxes)} items on this page.")
    
    extracted_items = []
    
    # 2. Use 'zip' to pair the 1st name with the 1st price, 2nd with 2nd, etc.
    for name_box, price_box in zip(name_boxes, price_boxes):
        raw_name = name_box.text.strip()
        raw_price = price_box.text.strip()
        
        # 3. Clean the price using your logic!
        clean_text = raw_price.replace('$', '').replace(',', '').replace('CAD', '').strip()
        
        if clean_text:
            clean_price = float(clean_text)
            
            # 4. Save the paired name and price together as a dictionary
            extracted_items.append({
                "name": raw_name,
                "price": clean_price
            })
            print(f"🛍️ Found: {raw_name} : ${clean_price}")
            
    return extracted_items

# Let's test it with our html_data!
if html_data:
    all_items = extract_items_from_list(html_data)

🥣 Giving the blueprint to BeautifulSoup...
🎯 Jackpot! Found 1 items on this page.
🛍️ Found: Black Small Croissant Bag : $775.0


In [12]:
from bs4 import BeautifulSoup

def extract_items_from_list(html_data):
    print("🥣 Giving the blueprint to BeautifulSoup...")
    soup = BeautifulSoup(html_data, 'html.parser')
    
    # 1. THE MAGIC FIX: Use 'select' with '^=' which means "starts with"
    # Find all spans where data-test starts with "productName"
    name_boxes = soup.select('span[data-test^="productName"]')
    price_boxes = soup.select('span[data-test^="productCurrentPrice"]')
    
    if not name_boxes or not price_boxes:
        print("⚠️ Couldn't find items on the shelf.")
        return []
        
    print(f"🎯 Jackpot! Found {len(name_boxes)} items on this page.")
    
    extracted_items = []
    
    # 2. Pair them up and clean the price just like before
    for name_box, price_box in zip(name_boxes, price_boxes):
        raw_name = name_box.text.strip()
        raw_price = price_box.text.strip()
        
        clean_text = raw_price.replace('$', '').replace(',', '').replace('CAD', '').strip()
        
        if clean_text:
            clean_price = float(clean_text)
            extracted_items.append({
                "name": raw_name,
                "price": clean_price
            })
            print(f"🛍️ Found: {raw_name} : ${clean_price}")
            
    return extracted_items

# Let's test the fixed extractor!
if html_data:
    all_items = extract_items_from_list(html_data)

🥣 Giving the blueprint to BeautifulSoup...
🎯 Jackpot! Found 2 items on this page.
🛍️ Found: Black Small Croissant Bag ---> $1220.0
🛍️ Found: Black Small Croissant Bag ---> $775.0
